In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm
from joblib import Parallel, delayed
import os

from torch_geometric.data import Dataset, Data
import torch

In [2]:
#Edge Augmentation - Set to 1.0 for no augmentation or larger for factor increase
augment = 1.0

datadir = 'data'
dstdir = 'data_GNN'
cwd = os.getcwd()

path = os.path.join(cwd, datadir)
dstpath = os.path.join(cwd, dstdir)
files = os.listdir(path)

In [3]:
#list of jobs that ran
info = []

for file in files:
    #Not all inputs ran successfully; filter on results
    if file.endswith('.csv') and 'processed' not in file:
        info.append(file)
        continue

In [6]:
def read_nsets(mesh, midside):
    src = open(mesh, 'r')
    lines = src.readlines()
    src.close()

    nsets = {}
    nflag = 0

    #get nset lines, store in dict
    for line in lines:
        line = line.strip()
        if line.upper().startswith('*NSET, '):
            temp = line.split('=')[1]
            name = temp.split(',')[0]

            #skip some NSETS
            skiplist = ['Force', 'RefPt', 'Beam-1_Beam']
            if name in skiplist:
                continue
            eflag = 1
            nsets[name] = {'text':[], 'nodes':[]}
            continue

        if line.upper().startswith('*'):
            eflag = 0
            continue

        if eflag == 1:
            line = line.strip()
            nsets[name]['text'].append(line)

    #parse nset lines into list of node numbers
    #split node lines of nset, save unless midside node
    for key in nsets.keys():
        lines = nsets[key]['text']
        for line in lines:
            line = line.strip()
            nodes = line.split(',')
            for node in nodes:
                if node == '':
                    continue
                node = int(node.strip())
                if node not in midside:
                    nsets[key]['nodes'].append(node)
    return nsets

In [7]:
#Funciton to find smallest dim in list of ndoes
#useful for finding boundaries

def get_min_y(nodenums, df):
    fymin = 99999
    for node in nodenums:
        #retrieve y-coord
        #retrieve value from array from to_numpy()
        ytemp = df[df['node'] == node]['ycoord'].to_numpy()[0]
        if ytemp < fymin:
            fymin = ytemp
    return fymin

def get_max_x(nodenums, df):
    fxmax = -99999
    for node in nodenums:
        #retrieve xcoord
        #retrieve value from array from to_numpy()
        xtemp = df[df['node'] == node]['xcoord'].to_numpy()[0]
        if xtemp > fxmax:
            fxmax = xtemp
    return fxmax

In [8]:
#Assign interger to nodes based on type (edge, force, etc); one-hot later
def encode_nodes(val, df, fillet_xmax, fillet_ymin):
    '''
    0: Fixed BC
    1: Loaded edge
    2: Fillet
    2: Hole
    2: Bottom Edge
    2: Top edge and top edge past fillet
    3. Interior
    '''

    #Check for types 0-2 and update
    ntype = 3 

    if val['xcoord'] == df.xcoord.min(): #left edge
        ntype = 0
    elif val['xcoord'] == df.xcoord.max(): #right edge
        ntype = 1
    elif val['node'] in nsets['Hole']['nodes']: #Hole
        ntype = 2 
    elif val['node'] in nsets['Fillet']['nodes']: #Fillet
        ntype = 2
    elif val['ycoord'] == df.ycoord.min(): #Bottom edge
        ntype = 2
    elif  val['ycoord'] == df.ycoord.max(): #Top edge
        ntype = 2
    elif val['ycoord'] > fillet_xmax and abs(val['ycoord'] - fillet_ymin) < 0.005: #top edge beyond fillet
        ntype = 2

    return ntype

In [24]:
def edgedist(val, df):
    #calculate distance to nearest edge
    xcoord = val['xcoord']
    ycoord = val['ycoord']
    type = val['NodeType']

    if type == 3:
        #get x, y coords of all edge nodes
        edges = df[df['NodeType'] !=3][['xcoord', 'ycoord']].to_numpy()
        distances = np.sqrt((edges[:,0] - xcoord)**2 + (edges[:,1] - ycoord)**2)
        idx = np.argmin(distances)
        #components, resultant
        val['xdist'] = edges[idx, 0] - xcoord
        val['ydist'] = edges[idx, 1] - ycoord
        val['EdgeDist'] = distances.min()

    else:
        #components, resultant
        val['xdist'] = 0.
        val['ydist'] = 0.
        val['EdgeDist'] = 0.

    return val

In [25]:
def read_elements(jobname):
    
    src = open(jobname, 'r')
    lines = src.readlines()
    src.close()

    corner = []
    midside = []

    eflag=0
    for line in lines:
        line = line.strip()
        if line.upper().startswith('*ELEMENT, '):
            eflag=1
            continue

        if line.upper().startswith('*'):
            eflag=0
            continue

        if eflag == 1:
            corner.append(np.array(line.split(',')[1:5], dtype='uint16'))
            midside.append(np.array(line.split(',')[5:], dtype='uint16'))

    corner = np.array(corner)   
    midside = np.array(midside) 
    return corner, midside

In [26]:
def adjacency(node, elements, nodelist, augment, df):
    idx, cols = np.where(elements == node)

    connections = np.array([])
    for id in idx:
        connections = np.concatenate((connections, elements[id]))

    unique = np.unique(connections)
    adjacent = np.delete(unique, np.where(unique==node)[0][0])

    #Augmentation - additinal nodes far away
    target = int(augment * adjacent.shape[0])
    while adjacent.shape[0] < target:
        idx = np.random.randint(0, len(nodelist))
        nnum = nodelist[idx]

        #try again if aleady in adjacency or is source node
        if nnum in adjacent or nnum == node:
            continue
            
        #distance calc - must be greater than half model size away
        dx = df[df.node == node].xcoord.to_numpy()[0] - df[df.node == nnum].xcoord.to_numpy()[0]
        dy = df[df.node == node].ycoord.to_numpy()[0] - df[df.node == nnum].ycoord.to_numpy()[0]
        dist = np.sqrt(dx**2 + dy**2)
        
        xmax = max(df.xcoord.tolist())
        ymax = max(df.ycoord.tolist())
        criteria = max(xmax, ymax) / 2
        if dist < criteria:
            continue

        #append if new
        adjacent = np.append(adjacent, nnum)

    #set to index from node number
    adjacent = adjacent - 1
    source = np.ones_like(adjacent) * node - 1

    return [source.astype(np.uint16), adjacent.astype(np.int16)]

In [27]:
def edgeattr(val, xcol, ycol, df):
    # ############3
    # val = df.iloc[[0],-1][0]
    # xcol = 1
    # ycol = 2
    # ############3
    dxs = np.array([])
    dys = np.array([])
    norms = np.array([])

    #Loop over node1_idx, node2_idx pairs in adjacency
    for idx1, idx2 in zip(val[0], val[1]):
        xy1 = np.array([df.iloc[idx1, xcol], df.iloc[idx1, ycol]])
        xy2 = np.array([df.iloc[idx2, xcol], df.iloc[idx2, ycol]])

        #Deltas
        dxy = xy2 - xy1
        dxs=np.concatenate((dxs, [dxy[0]]))
        dys=np.concatenate((dys, [dxy[1]]))

        #Euclidean distance
        norms=np.concatenate((norms, [np.linalg.norm(xy2-xy1)]))

    return [dxs.astype(np.float64), dys.astype(np.float64), norms.astype(np.float64)]

In [36]:
def prep_data(df):
    edge1 = []
    edge2 = []
    attrdx = []
    attrdy = []
    attrnorm = []

    for i, row in df.iterrows():
        edge1.extend(list(row['Adjacency'][0]))
        edge2.extend(list(row['Adjacency'][1]))
        
        attrdx.extend(list(row['Edge_Attr'][0]))
        attrdy.extend(list(row['Edge_Attr'][1]))
        attrnorm.extend(list(row['Edge_Attr'][2]))

    edge1 = np.array(edge1)
    edge2 = np.array(edge2)

    attrdx = np.array(attrdx).astype(np.float64)
    attrdy = np.array(attrdy).astype(np.float64)
    attrnorm = np.array(attrnorm).astype(np.float64)

    xlist = ['xcoord', 'ycoord', 'xinv', 'yinv', 'EdgeDist', 'xdist', 'ydist']
    columns = df.columns.to_list()

    #one-hot columns for nodetype
    for col in columns:
        if 'ntype' in col:
            xlist.append(col)

    x = df[xlist].to_numpy()
    y = df[['S11', 'S22', 'S12', 'Mises']].to_numpy()
    
    x = torch.from_numpy(x.astype(np.float64))
    y = torch.from_numpy(y.astype(np.float64))
    edge_index = torch.tensor(np.array([edge1, edge2]), dtype=torch.long)
    edge_attr = torch.tensor(np.array([attrdx, attrdy, attrnorm]).T)
    
    data = Data(x=x, y=y, edge_index=edge_index, edge_attr=edge_attr)
    return data

In [29]:
def remove_midside_rows(df, midside):
    #convert to 1d
    midside = midside.flatten()
    #remove duplicates
    midside = list(set(midside))

    #get indices of midside nodes in df
    nodes = df.node.to_list()
    idx = []
    for ms in midside:
        idx.append(nodes.index(ms))

    df.drop(idx, inplace=True)
    return df        

In [38]:
def process_files(file, testflag):
    global df, nsets, cornernodes, midside
    
    jobname = file.split('.')[0] 

    #mesh info
    jobfile = os.path.join(path, jobname+'.inp')
    cornernodes, midside = read_elements(jobfile)

    #ignore midside nodes
    nsets = read_nsets(jobfile, midside)

    #read CSV with nodes, coords, stresses
    csvfile = os.path.join(path, file)
    df = pd.read_csv(csvfile, index_col=None)
    df = remove_midside_rows(df, midside)

    #fillet bounds
    fillet_ymin = get_min_y(nsets['Fillet']['nodes'], df)
    fillet_xmax = get_max_x(nsets['Fillet']['nodes'], df)

    #Column indices for reference
    columns = df.columns
    xcol = list(columns).index('xcoord')
    ycol = list(columns).index('ycoord')

    nodelist = df.node.to_list()
    df['Adjacency'] = df['node'].apply(adjacency, args=(cornernodes, nodelist, augment, df))
    df['Edge_Attr'] = df['Adjacency'].apply(edgeattr, args=(xcol, ycol, df))

    #Node types
    df['NodeType'] = df[['node', 'xcoord', 'ycoord']].apply(encode_nodes, args=(df, fillet_xmax, fillet_ymin,), axis=1)
    dftemp = pd.get_dummies(df['NodeType'], prefix='ntype')

    #merge df, dftemp
    df = pd.concat([df, dftemp], axis = 1)

    #Distance to edge for interior nodes
    df = df.apply(edgedist, args=(df,), axis=1)

    #Create xinv, yinv
    df['xinv'] = df.xcoord.max() - df.xcoord
    df['yinv'] = df.ycoord.max() - df.ycoord

    #save csv
    processedfile = os.path.join(dstpath, jobname + '_processed.csv')
    df.to_csv(processedfile)

    #PyTorch data file
    data = prep_data(df)
    datafile = os.path.join(dstpath, jobname + '.pt')
    torch.save(data, datafile)

    if testflag == 1:
        return data

In [39]:
# #standalone
# process_files(info[0], 0)

In [41]:
Parallel(n_jobs=6, verbose=20)(delayed(process_files)(file, 0) for file in info)

[Parallel(n_jobs=6)]: Using backend LokyBackend with 6 concurrent workers.
[Parallel(n_jobs=6)]: Done   1 tasks      | elapsed:    2.5s
[Parallel(n_jobs=6)]: Done   2 tasks      | elapsed:    2.8s
[Parallel(n_jobs=6)]: Done   3 tasks      | elapsed:    4.1s
[Parallel(n_jobs=6)]: Done   4 tasks      | elapsed:    4.2s
[Parallel(n_jobs=6)]: Done   5 tasks      | elapsed:    4.3s
[Parallel(n_jobs=6)]: Done   6 tasks      | elapsed:    4.6s
[Parallel(n_jobs=6)]: Done   7 tasks      | elapsed:    5.2s
[Parallel(n_jobs=6)]: Done   8 tasks      | elapsed:    6.6s
[Parallel(n_jobs=6)]: Done   9 tasks      | elapsed:    7.3s
[Parallel(n_jobs=6)]: Done  10 tasks      | elapsed:    8.2s
[Parallel(n_jobs=6)]: Done  11 tasks      | elapsed:    8.4s
[Parallel(n_jobs=6)]: Done  12 tasks      | elapsed:    8.9s
[Parallel(n_jobs=6)]: Done  13 tasks      | elapsed:    9.1s
[Parallel(n_jobs=6)]: Done  14 tasks      | elapsed:   11.2s
[Parallel(n_jobs=6)]: Done  15 tasks      | elapsed:   11.3s
[Parallel(

[None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,
 None,